# Seq2seq LSTM với Luong attention

TV2 ghép encoder, decoder và kiểm tra attention trên batch có padding.

In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/corpus.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from scripts.corpus import records

sample = next(records("train"))
print(sample)

In [ ]:
import torch

def luong_attention(query, encoder_states, valid_source):
    scores = torch.bmm(encoder_states, query.unsqueeze(2)).squeeze(2)
    weights = scores.masked_fill(~valid_source, float("-inf")).softmax(dim=-1)
    context = torch.bmm(weights.unsqueeze(1), encoder_states).squeeze(1)
    return context, weights

states = torch.randn(2, 4, 64)
valid = torch.tensor([[True, True, True, False], [True] * 4])
context, weights = luong_attention(torch.zeros(2, 64), states, valid)
assert weights[0, -1] == 0 and torch.allclose(weights.sum(1), torch.ones(2))

## Các ô cần hoàn thiện

Encoder embedding + LSTM dùng độ dài thật. Decoder nhận token trước, cập nhật trạng thái, ghép context với hidden và chiếu lên vocab. Dịch chuyển nhãn một vị trí, bỏ PAD khỏi loss. Lưu attention từng bước trước EOS. Dùng lại hàm decode_step cho greedy và beam.

In [ ]:
config = dict(embedding_dim=256, hidden_size=256, layers=2, dropout=0.2, seed=42)
tokenizers = ["word", "bpe", "unigram"]